In [ ]:
# !pip install transformers
# !pip install jiwer
# !pip install torch
# !pip install Pillow
# !pip install kagglehub
# !pip install IPython
# !pip install scikit-learn
# !pip install pandas
# !pip install tqdm
# !pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
# !pip install torchvision
# !pip install openpyxl


In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torchvision import transforms
from PIL import Image
import requests
import kagglehub
from IPython.display import display
from sklearn.model_selection import train_test_split
import pandas as pd
import os
from tqdm import tqdm
from jiwer import cer, wer
import torch.nn.utils.rnn as rnn_utils
from transformers import get_linear_schedule_with_warmup
import logging
import csv
from datetime import datetime
import shutil

# Датасет заказчика

## Объединение датасета

In [ ]:
import re

def replace_folder(path, correct):
    return re.sub(r'IMG_[\d_]+S', correct, path)

In [ ]:
path = 'C:/Users/maill/WORK/Dataset' # указываем тут путь

all_data = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(('.xlsx')):
            # print(os.path.join(root, file))
            
            full_path = os.path.join(root, file)
            if file.endswith(('.xlsx')):
                df = pd.read_excel(full_path)
            # else:
            #     df = pd.read_csv(full_path, sep=';', on_bad_lines='skip', encoding='utf-8-sig', encoding_errors='replace')

            df['image_path'] = df['image_path'].apply(lambda x: os.path.join(x.replace('\\', '/')) if pd.notna(x) else x)

            #Правим опечатки в путях
            
            name = os.path.basename(full_path)

            df['image_path'] = str(os.path.splitext(name)[0]) + '/' + df['image_path'].str.split('/').str[1]

            df['image_path'] = root + '/' + df['image_path']

            all_data.append(df)

In [ ]:
final_df = pd.concat(all_data, ignore_index=True)

In [ ]:
final_df = final_df.drop('Unnamed: 0', axis=1)

In [ ]:
print(list(final_df))

In [ ]:
# final_df[final_df['label'].isna()]
final_df = final_df.dropna(subset=['image_path'])

In [ ]:
print(f'Количество labels = {len(final_df)}')
print('Пустых label:', final_df['label'].isna().sum())
print('Заполненных:', final_df['label'].notna().sum())

# Посмотрим картинки

In [ ]:
img_path = final_df[final_df['label'].notna()].iloc[0]['image_path']
print(img_path)
image = Image.open(img_path + '.jpg').convert('RGB')
display(image)

# Подготавливаем данные

In [ ]:
train_df, temp_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=None)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=None)#!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
train_images = train_df['image_path'].tolist()
train_texts = train_df['label'].astype(str).tolist()

val_images = val_df['image_path'].tolist()
val_texts = val_df['label'].astype(str).tolist()

test_images = test_df['image_path'].tolist()
test_texts = test_df['label'].astype(str).tolist()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
processor = TrOCRProcessor.from_pretrained('kazars24/trocr-base-handwritten-ru')
model = VisionEncoderDecoderModel.from_pretrained('kazars24/trocr-base-handwritten-ru')

model.to(device)

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id

In [ ]:
class CustomerDataset(Dataset):
    def __init__(self, image_paths, texts, processor, augment=False):
        self.image_paths = image_paths
        self.texts = texts
        self.processor = processor

        if augment:
            self.augmentation = transforms.Compose([
                transforms.RandomRotation(degrees=2),
                transforms.RandomAffine(degrees=0, translate=(0.03, 0.03)),
                transforms.ColorJitter(brightness=0.3, contrast=0.3),
            ])
        else:
            self.augmentation = None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        base_path = str(self.image_paths[idx])
        text = str(self.texts[idx])
        
        if os.path.exists(base_path):
            img_path = base_path
        else:
            img_path = base_path + '.jpg'
            if not os.path.exists(img_path):
                img_path = base_path + '.png'
                if not os.path.exists(img_path):
                    raise FileNotFoundError(f"Изображение не найдено: {base_path}")
        
        img = Image.open(img_path).convert('RGB')

        if self.augmentation:
            img = self.augmentation(img)

        pixel_values = self.processor(images=img, return_tensors='pt').pixel_values[0]
        labels = self.processor(text=text, return_tensors='pt').input_ids[0]

        return {'pixel_values': pixel_values, 'labels': labels}

In [ ]:
def customer_collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = [item['labels'] for item in batch]
    
    labels_padded = rnn_utils.pad_sequence(
        labels,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id
    )
    
    return {'pixel_values': pixel_values, 'labels': labels_padded}

In [ ]:
train_dataset_customer = CustomerDataset(train_images, train_texts, processor, augment=True)
val_dataset_customer = CustomerDataset(val_images, val_texts, processor, augment=False)
test_dataset_customer = CustomerDataset(test_images, test_texts, processor, augment=False)

In [ ]:
batch_size = 8 

train_loader = DataLoader(
    train_dataset_customer, 
    batch_size=batch_size, 
    shuffle=True, 
    collate_fn=customer_collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset_customer, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=customer_collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset_customer, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=customer_collate_fn,
    num_workers=0
)

model.config.decoder.dropout = 0.05
model.config.encoder.hidden_dropout_prob = 0.05


In [ ]:
os.makedirs("./training_logs", exist_ok=True)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("./training_logs/training.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Обучаем (на данных заказчика)

In [ ]:
LR = 2e-6
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
EPOCHS = 4
accumulation_steps = 2 

num_training_steps = len(train_loader) * EPOCHS // accumulation_steps
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [ ]:
logger.info("="*60)
logger.info("НАЧАЛО ОБУЧЕНИЯ НА ДАННЫХ ЗАКАЗЧИКА")
logger.info("="*60)
logger.info(f"Размер обучающей выборки: {len(train_dataset_customer)}")
logger.info(f"Размер валидационной выборки: {len(val_dataset_customer)}")
logger.info(f"Batch size: {batch_size}")
logger.info(f"Epochs: {EPOCHS}")
logger.info(f"Learning rate: {LR}")
logger.info(f"Device: {device}")

In [ ]:
metrics_log = []
csv_filename = "./training_logs/metrics.csv"

In [ ]:
best_val_cer = float('inf')
best_val_wer = float('inf')
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    epoch_start_time = datetime.now()
    # Train
    model.train()
    train_loss_total = 0
    optimizer.zero_grad()
    
    with tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]") as pbar:
        for i, batch in enumerate(pbar):
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / accumulation_steps
            loss.backward()
            
            train_loss_total += outputs.loss.item() / accumulation_steps
            
            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            
            pbar.set_postfix({'loss': f'{outputs.loss.item():.4f}'})
            
            del pixel_values, labels, outputs, loss
    
    avg_train_loss = train_loss_total / len(train_loader)
    
    # Validation
    model.eval()
    val_loss_total = 0
    val_cer_scores = []
    val_wer_scores = []
    
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]") as pbar:
            for batch in pbar:
                pixel_values = batch['pixel_values'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(pixel_values=pixel_values, labels=labels)
                val_loss_total += outputs.loss.item()
                
                generated_ids = model.generate(
                    pixel_values, 
                    max_length=64,
                    num_beams=2,
                    early_stopping=True
                )
                
                pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
                true_texts = processor.batch_decode(labels, skip_special_tokens=True)
                
                for pred, true in zip(pred_texts, true_texts):
                    val_cer_scores.append(cer(true, pred))
                    val_wer_scores.append(wer(true, pred))
                
                pbar.set_postfix({
                    'val_loss': f'{outputs.loss.item():.4f}',
                    'val_CER': f'{sum(val_cer_scores[-len(pred_texts):])/len(pred_texts):.4f}',
                    'val_WER': f'{sum(val_wer_scores[-len(pred_texts):])/len(pred_texts):.4f}'
                })
                
                del pixel_values, labels, outputs, generated_ids

    epoch_end_time = datetime.now()
    epoch_duration = (epoch_end_time - epoch_start_time).total_seconds()
    
    avg_val_loss = val_loss_total / len(val_loader)
    avg_val_cer = sum(val_cer_scores) / len(val_cer_scores)
    avg_val_wer = sum(val_wer_scores) / len(val_wer_scores)

    logger.info(f"Epoch {epoch+1}/{EPOCHS} завершена за {epoch_duration:.1f} сек")
    logger.info(f"  Train Loss: {avg_train_loss:.4f}")
    logger.info(f"  Val Loss: {avg_val_loss:.4f}")
    logger.info(f"  Val CER: {avg_val_cer:.4f}")
    logger.info(f"  Val WER: {avg_val_wer:.4f}")
    
    print(f"Epoch {epoch+1}/{EPOCHS}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val CER={avg_val_cer:.4f}, Val WER={avg_val_wer:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_val_cer = avg_val_cer
        best_val_wer = avg_val_wer
        model.save_pretrained("./FINAL_MODEL")
        processor.save_pretrained("./FINAL_MODEL")
        print(f" Сохранена лучшая модель (CER={avg_val_cer:.4f}), WER={avg_val_wer:.4f})")

print(f"Обучение завершено. Лучшие CER: {best_val_cer:.4f}, WER: {best_val_wer:.4f}")

# Тестирование 

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained("./FINAL_MODEL")
processor = TrOCRProcessor.from_pretrained("./FINAL_MODEL")
model.to(device)
model.eval()

In [ ]:
test_cer_scores = []
test_wer_scores = []

test_loss_total = 0

with torch.no_grad():
    with tqdm(test_loader, desc="Testing") as pbar:
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            
            generated_ids = model.generate(pixel_values, max_length=64, num_beams=2)
            outputs = model(pixel_values=pixel_values, labels=labels)
            test_loss_total += outputs.loss.item()
            
            pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            true_texts = processor.batch_decode(labels, skip_special_tokens=True)
            
            for pred, true in zip(pred_texts, true_texts):
                test_cer_scores.append(cer(true, pred))
                test_wer_scores.append(wer(true, pred))
            
            pbar.set_postfix({
                'test_loss': f'{outputs.loss.item():.4f}',
                'test_CER': f'{sum(test_cer_scores[-len(pred_texts):])/len(pred_texts):.4f}',
                'test_WER': f'{sum(test_wer_scores[-len(pred_texts):])/len(pred_texts):.4f}'
            })
           
            del pixel_values, labels, generated_ids, outputs

avg_test_loss = test_loss_total / len(test_loader)
avg_test_cer = sum(test_cer_scores) / len(test_cer_scores)
avg_test_wer = sum(test_wer_scores) / len(test_wer_scores)

print(f"\n=== РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ ===")
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Average CER: {avg_test_cer:.4f}")
print(f"Average WER: {avg_test_wer:.4f}")

In [ ]:
def predict_batch(image_paths, model, processor, device):
    results = []
    for img_path in tqdm(image_paths):
        try:
            image = Image.open(img_path).convert('RGB')

            display(image)

            pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)
            generated_ids = model.generate(pixel_values, max_length=64, num_beams=2)
            text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            results.append({'image_path': img_path, 'predicted_text': text})
        except Exception as e:
            print(f"Ошибка на {img_path}: {e}")
            results.append({'image_path': img_path, 'predicted_text': '', 'error': str(e)})
    
    return pd.DataFrame(results)

In [ ]:
image_paths = [
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_10_30_17S/img_1.png",
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_10_30_17S/img_81.png",
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_11_02_54S/img_16.png",
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_11_02_54S/img_21.png",
    "C:/Users/maill/WORK/Dataset/5_1/img (5).jpg",
    "C:/Users/maill/WORK/Dataset/5_1/img (35).jpg",
    "C:/Users/maill/WORK/Dataset/Text_5/img_45.png",
    "C:/Users/maill/WORK/Dataset/Text_2/img_49.png",
]

df_results = predict_batch(image_paths, model, processor, device)
print(df_results)

In [ ]:
final_df.to_csv('FINAL_DATAFRAME.csv', index=False, encoding='utf-8-sig')

# Дообучение на кагл данных + заказчика

In [ ]:
logger.info("="*60)
logger.info("ЗАГРУЗКА И ПОДГОТОВКА KAGGLE ДАТАСЕТА")
logger.info("="*60)

In [ ]:
kaggle_path = kagglehub.dataset_download("constantinwerner/cyrillic-handwriting-dataset")

In [ ]:
img_path = 'C:\\Users\\maill\\.cache\\kagglehub\\datasets\\constantinwerner\\cyrillic-handwriting-dataset\\versions\\5\\train\\aa1.png'
image = Image.open(img_path).convert('RGB')
display(image)

In [ ]:
df_train_kaggle = pd.read_csv(os.path.join(kaggle_path, "train.tsv"), sep='\t', header=None, names=['image', 'text'])
df_test_kaggle = pd.read_csv(os.path.join(kaggle_path, "test.tsv"), sep='\t', header=None, names=['image', 'text'])
df_kaggle = pd.concat([df_train_kaggle, df_test_kaggle], ignore_index=True)

In [ ]:
print(f"Train samples: {len(df_train_kaggle)}")
print(f"Test samples: {len(df_test_kaggle)}")
print(f"Total samples: {len(df_kaggle)}")
print(df_kaggle.head())

In [ ]:
def find_img(fname):
    for folder in ['train', 'test']:
        path = os.path.join(kaggle_path, folder, fname)
        if os.path.exists(path): return path
    return None

df_kaggle['image_path'] = df_kaggle['image'].apply(find_img)
df_kaggle['label'] = df_kaggle['text'].astype(str)
df_kaggle = df_kaggle.dropna(subset=['image_path'])


In [ ]:
N_KAGGLE = 20000
df_kaggle_sample = df_kaggle.sample(n=min(N_KAGGLE, len(df_kaggle)), random_state=42)
logger.info(f"Выбрано {len(df_kaggle_sample)} образцов из Kaggle для дообучения")

In [ ]:
train_df_combined = pd.concat([train_df, df_kaggle_sample[['image_path', 'label']]], ignore_index=True)
train_df_combined = train_df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
logger.info(f"Итоговый Train: {len(train_df_combined)} | Val (заказчик): {len(val_df)} | Test (заказчик): {len(test_df)}")

In [ ]:
train_images = train_df_combined['image_path'].tolist()
train_texts = train_df_combined['label'].astype(str).tolist()
val_images = val_df['image_path'].tolist()
val_texts = val_df['label'].astype(str).tolist()
test_images = test_df['image_path'].tolist()
test_texts = test_df['label'].astype(str).tolist()

In [ ]:
logger.info("Загрузка весов модели после первого этапа обучения")
model = VisionEncoderDecoderModel.from_pretrained("./FINAL_MODEL")
processor = TrOCRProcessor.from_pretrained("./FINAL_MODEL")
model.to(device)

In [ ]:
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id

## Создаю датасеты

In [ ]:
train_dataset_combined = CustomerDataset(train_images, train_texts, processor, augment=True)
val_dataset_customer = CustomerDataset(val_images, val_texts, processor, augment=False)
test_dataset_customer = CustomerDataset(test_images, test_texts, processor, augment=False)

In [ ]:
BATCH_SIZE = 8         
ACCUMULATION_STEPS = 1  
NUM_WORKERS = 0        
LR = 2e-5               
EPOCHS = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

num_training_steps = len(train_loader) * EPOCHS // ACCUMULATION_STEPS if 'train_loader' in locals() else len(DataLoader(train_dataset_combined, batch_size=BATCH_SIZE)) * EPOCHS // ACCUMULATION_STEPS
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps)

In [ ]:
train_loader = DataLoader(
    train_dataset_combined, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=customer_collate_fn, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset_customer, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=customer_collate_fn, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset_customer, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=customer_collate_fn, num_workers=NUM_WORKERS, pin_memory=True
)

In [ ]:
logger.info("="*60)
logger.info("НАЧАЛО ВТОРОГО ЭТАПА ОБУЧЕНИЯ (Заказчик + Kaggle)")
logger.info("="*60)
logger.info(f"Размер обучающей выборки: {len(train_dataset_combined)}")
logger.info(f"Размер валидационной выборки: {len(val_dataset_customer)}")
logger.info(f"Learning rate: {LR}")

In [ ]:
best_val_cer = float('inf')
best_val_wer = float('inf')
save_dir = "./FINAL_MODEL_COMBINED"
os.makedirs(save_dir, exist_ok=True)


import gc
gc.collect()
torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    epoch_start = datetime.now()
    model.train()
    train_loss_total = 0
    optimizer.zero_grad()

    with tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]") as pbar:
        for i, batch in enumerate(pbar):
            pixel_values = batch['pixel_values'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / ACCUMULATION_STEPS
            loss.backward()

            train_loss_total += outputs.loss.item() / ACCUMULATION_STEPS

            if (i + 1) % ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

                pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})
                
            del pixel_values, labels, outputs, loss

    avg_train_loss = train_loss_total / len(train_loader)

    model.eval()
    val_loss_total = 0
    val_cer_scores = []
    val_wer_scores = []

    torch.cuda.empty_cache()

    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]") as pbar:
            for batch in pbar:
                pixel_values = batch['pixel_values'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)

                outputs = model(pixel_values=pixel_values, labels=labels)
                val_loss_total += outputs.loss.item()

                generated_ids = model.generate(pixel_values, max_length=64, num_beams=2, early_stopping=True)
                pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
                true_texts = processor.batch_decode(labels, skip_special_tokens=True)

                for p, t in zip(pred_texts, true_texts):
                    val_cer_scores.append(cer(t, p))
                    val_wer_scores.append(wer(t, p))

                pbar.set_postfix({'val_CER': f'{sum(val_cer_scores)/len(val_cer_scores):.4f}'})
                
                del pixel_values, labels, outputs, generated_ids

    epoch_duration = (datetime.now() - epoch_start).total_seconds()
    avg_val_loss = val_loss_total / len(val_loader)
    avg_val_cer = sum(val_cer_scores) / len(val_cer_scores)
    avg_val_wer = sum(val_wer_scores) / len(val_wer_scores)

    logger.info(f"Epoch {epoch+1}/{EPOCHS} | Time: {epoch_duration:.1f}s | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val CER: {avg_val_cer:.4f} | Val WER: {avg_val_wer:.4f}")

    if avg_val_cer < best_val_cer:
        best_val_cer = avg_val_cer
        best_val_wer = avg_val_wer
        model.save_pretrained(save_dir)
        processor.save_pretrained(save_dir)
        logger.info(f"Сохранена лучшая модель в {save_dir} (CER={avg_val_cer:.4f}, WER={avg_val_wer:.4f})")

logger.info(f"Обучение завершено. Лучшие метрики на данных заказчика: CER: {best_val_cer:.4f}, WER: {best_val_wer:.4f}")

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(save_dir)
processor = TrOCRProcessor.from_pretrained(save_dir)
model.to(device)
model.eval()

test_cer_scores = []
test_wer_scores = []
with torch.no_grad():
    with tqdm(test_loader, desc="Testing") as pbar:
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            generated_ids = model.generate(pixel_values, max_length=64, num_beams=2)
            pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            true_texts = processor.batch_decode(labels, skip_special_tokens=True)
            for p, t in zip(pred_texts, true_texts):
                test_cer_scores.append(cer(t, p))
                test_wer_scores.append(wer(t, p))
            pbar.set_postfix({'test_CER': f'{sum(test_cer_scores[-len(pred_texts):])/len(pred_texts):.4f}'})

print(f"\n=== РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ (Заказчик) ===")
print(f"Test CER: {sum(test_cer_scores)/len(test_cer_scores):.4f}")
print(f"Test WER: {sum(test_wer_scores)/len(test_wer_scores):.4f}")

In [ ]:
image_paths = [
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_10_30_17S/img_1.png",
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_10_30_17S/img_81.png",
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_11_02_54S/img_16.png",
    "C:/Users/maill/WORK/Dataset/IMG_2024_08_08_11_02_54S/img_21.png",
    "C:/Users/maill/WORK/Dataset/5_1/img (5).jpg",
    "C:/Users/maill/WORK/Dataset/5_1/img (35).jpg",
    "C:/Users/maill/WORK/Dataset/Text_5/img_45.png",
    "C:/Users/maill/WORK/Dataset/Text_2/img_49.png",
]

df_results = predict_batch(image_paths, model, processor, device)
print(df_results)